In [ ]:
import pandas as pd
import os

# --- Configurazioni ---
LAMBDAS = [0.0, 0.2, 0.4, 0.6, 0.8, 1.0]
GMM_SCALES = [0.1, 0.5, 1.0, 2.0, 3.0, 4.0, 5.0]
IB_BETAS = [0.01, 0.05, 0.1, 0.5, 1.0]
GAUSSIAN_TYPES = [("Combinatorial", "combinatorial"), ("Independent", "independent")]

def format_val(mean, ci):
    """Formatta la media e il CI nel formato LaTeX desiderato."""
    if pd.isna(mean) or pd.isna(ci):
        return "N/A"
    mean *= 100
    ci *= 100
    return f"{mean:.2f}\\% $\\pm$ {ci:.2f}\\%"

def extract_data_and_format(file_path, row_label, has_lambda=True):
    """Legge il CSV, filtra i dati e genera la riga LaTeX."""
    row_str = f"        {row_label} "
    
    try:
        df = pd.read_csv(file_path)
        # rimuovi le righe in cui lambda=="tilde"
        if has_lambda and 'lambda' in df.columns:
            df = df[df['lambda'] != 'tilde']
    except FileNotFoundError:
        # Se il file non esiste, riempie la riga con N/A per non interrompere lo script
        return row_str + "& N/A & N/A & N/A & N/A & N/A & N/A \\\\"

    if not has_lambda:
        mask = df['intervention_group'] == 'both'
        filtered = df[mask]
        
        if not filtered.empty:
            mean = filtered.iloc[0]['style_overall_mean']
            ci = filtered.iloc[0]['style_overall_ci_95']
            val_str = format_val(mean, ci)
        else:
            val_str = "N/A"
            
        # Usa doppie graffe {{ }} per l'escape in f-string
        row_str += f"& \\multicolumn{{6}}{{c}}{{{val_str}}} \\\\"
        return row_str

    for lam in LAMBDAS:
        # Filtro base
        mask = df['intervention_group'] == 'both'
        
        # Filtro per lambda se applicabile
        if has_lambda and 'lambda' in df.columns:
            # Arrotondamento per prevenire errori di floating point (es. 0.2000000000001)
            mask = mask & (df['lambda'].astype(float).round(2) == lam)
        
        filtered = df[mask]
        
        if not filtered.empty:
            mean = filtered.iloc[0]['style_overall_mean']
            ci = filtered.iloc[0]['style_overall_ci_95']
            val_str = format_val(mean, ci)
        else:
            val_str = "N/A"
            
        row_str += f"& {val_str} "
        
    row_str += "\\\\"
    return row_str

def generate_latex_table():
    latex_lines = [
        "\\begin{tabular}{l c c c c c c}",
        "        \\toprule",
        "        \\textbf{Model Configuration} & $\\lambda=0.0$ & $\\lambda=0.2$ & $\\lambda=0.4$ & $\\lambda=0.6$ & $\\lambda=0.8$ & $\\lambda=1.0$ \\\\",
        "        \\midrule",
        "        \\textbf{PMF-GMM} & & & \\\\"
    ]

    # --- 1. PMF-GMM ---
    for scale in GMM_SCALES:
        # Gestiamo il formato di scale se fosse intero o float nel nome file
        # Assumiamo che il nome file usi '0.1', '1.0' ecc. come stringa
        path = f"csv/GMM_Scale_{scale}/cyc_cons_GMM_{scale}_aggregate.csv"
        line = extract_data_and_format(path, f"Scale = {scale}", has_lambda=True)
        latex_lines.append(line)

    latex_lines.append("        \\midrule")
    latex_lines.append("        \\textbf{PMF-IB} & & & \\\\")

    # --- 2. PMF-IB ---
    for beta in IB_BETAS:
        path = f"csv/IB_Beta_{beta}/cyc_cons_IB_{beta}_aggregate.csv"
        line = extract_data_and_format(path, f"$\\beta$ = {beta}", has_lambda=True)
        latex_lines.append(line)

    latex_lines.append("        \\midrule")
    latex_lines.append("        \\textbf{HCE + ResNet} & & & \\\\")

    # --- 3. Gaussian (HCE + ResNet) ---
    for label, folder_type in GAUSSIAN_TYPES:
        path = f"csv/Gaussian/{folder_type}/cyc_cons_Gaussian_{folder_type}_aggregate.csv"
        # has_lambda=False in base alle istruzioni: prenderà l'unica riga e la ripeterà
        line = extract_data_and_format(path, label, has_lambda=False)
        latex_lines.append(line)

    latex_lines.append("        \\midrule")

    # --- 4. Conditional (CNF + ResNet) ---
    path_cond = "csv/Conditional/cyc_cons_Conditional_aggregate.csv"
    line_cond = extract_data_and_format(path_cond, "\\textbf{CNF + ResNet}", has_lambda=False)
    latex_lines.append(line_cond)

    latex_lines.append("        \\bottomrule")
    latex_lines.append("        \\end{tabular}%")

    return "\n".join(latex_lines)

if __name__ == "__main__":
    final_latex = generate_latex_table()
    print(final_latex)

\begin{tabular}{l c c c c c c}
        \toprule
        \textbf{Model Configuration} & $\lambda=0.0$ & $\lambda=0.2$ & $\lambda=0.4$ & $\lambda=0.6$ & $\lambda=0.8$ & $\lambda=1.0$ \\
        \midrule
        \textbf{PMF-GMM} & & & \\
        Scale = 0.1 & 100.00\% $\pm$ 0.00\% & 100.00\% $\pm$ 0.00\% & 99.65\% $\pm$ 0.18\% & 96.90\% $\pm$ 0.51\% & 83.37\% $\pm$ 1.33\% & 56.52\% $\pm$ 1.74\% \\
        Scale = 0.5 & 94.00\% $\pm$ 16.66\% & 93.05\% $\pm$ 19.29\% & 89.90\% $\pm$ 27.81\% & 85.83\% $\pm$ 36.79\% & 80.01\% $\pm$ 41.20\% & 69.32\% $\pm$ 39.27\% \\
        Scale = 1.0 & 100.00\% $\pm$ 0.00\% & 100.00\% $\pm$ 0.01\% & 99.80\% $\pm$ 0.12\% & 98.17\% $\pm$ 0.73\% & 91.73\% $\pm$ 2.77\% & 77.84\% $\pm$ 5.38\% \\
        Scale = 2.0 & 100.00\% $\pm$ 0.00\% & 99.95\% $\pm$ 0.12\% & 99.37\% $\pm$ 1.00\% & 97.11\% $\pm$ 2.24\% & 90.09\% $\pm$ 3.92\% & 76.00\% $\pm$ 5.20\% \\
        Scale = 3.0 & 100.00\% $\pm$ 0.00\% & 99.99\% $\pm$ 0.01\% & 99.77\% $\pm$ 0.18\% & 98.64\% $\pm$ 0.32

In [19]:
import pandas as pd
import os

# --- Configurazioni ---
LAMBDAS = [0.0, 0.2, 0.4, 0.6, 0.8, 1.0]
GMM_SCALES = [0.1, 1.0, 2.0, 3.0, 4.0, 5.0]
IB_BETAS = [0.01, 0.05, 0.1, 0.5, 1.0]
GAUSSIAN_TYPES = [("Combinatorial", "combinatorial"), ("Independent", "independent")]

def extract_data_and_format(file_path, row_label, has_lambda=True):
    """Legge il CSV, filtra i dati e genera la riga LaTeX."""
    row_str = f"        {row_label} "
    
    try:
        df = pd.read_csv(file_path)
    except FileNotFoundError:
        if not has_lambda:
            # Aggiornato a 7 colonne
            return row_str + "& \\multicolumn{7}{c}{N/A} \\\\"
        return row_str + "& N/A & N/A & N/A & N/A & N/A & N/A & N/A \\\\"

    if not has_lambda:
        mask = df['intervention_group'] == 'AGGREGATE_1_INTERVENTION'
        filtered = df[mask]
        
        if not filtered.empty:
            mean = filtered.iloc[0]['disc_overall_acc_mean']
            ci = filtered.iloc[0]['disc_overall_acc_ci_95']
            val_str = format_val(mean, ci)
        else:
            val_str = "N/A"
            
        # Aggiornato a 7 colonne
        row_str += f"& \\multicolumn{{7}}{{c}}{{{val_str}}} \\\\"
        return row_str

    # Iteriamo sui lambda numerici standard PIÙ 'tilde'
    for lam in LAMBDAS + ['tilde']:
        mask = df['intervention_group'] == 'AGGREGATE_1_INTERVENTION'
        
        if 'lambda' in df.columns:
            if lam == 'tilde':
                # Ricerca stringa esatta per tilde
                mask = mask & (df['lambda'].astype(str) == 'tilde')
            else:
                # Conversione sicura: trasforma 'tilde' in NaN durante il check 
                # numerico per evitare ValueError
                numeric_lambda = pd.to_numeric(df['lambda'], errors='coerce')
                mask = mask & (numeric_lambda.round(2) == lam)
        
        filtered = df[mask]
        
        if not filtered.empty:
            mean = filtered.iloc[0]['disc_overall_acc_mean']
            ci = filtered.iloc[0]['disc_overall_acc_ci_95']
            val_str = format_val(mean, ci)
        else:
            val_str = "N/A"
            
        row_str += f"& {val_str} "
        
    row_str += "\\\\"
    return row_str

def generate_latex_table():
    latex_lines = [
        "\\begin{tabular}{l c c c c c c | c}",
        "        \\toprule",
        "        \\textbf{Model} & $\\lambda=0.0$ & $\\lambda=0.2$ & $\\lambda=0.4$ & $\\lambda=0.6$ & $\\lambda=0.8$ & $\\lambda=1.0$ & $\\lambda=\\tilde{\\lambda}$\\\\",
        "        \\midrule",
        "        \\textbf{PMF-GMM} & & & & & & & \\\\"
    ]

    # --- 1. PMF-GMM ---
    for scale in GMM_SCALES:
        path = f"csv/GMM_Scale_{scale}/cyc_cons_GMM_{scale}_aggregate.csv"
        line = extract_data_and_format(path, f"Scale = {scale}", has_lambda=True)
        latex_lines.append(line)

    latex_lines.append("        \\midrule")
    # Aggiunti gli & mancanti per allineare correttamente l'intestazione PMF-IB
    latex_lines.append("        \\textbf{PMF-IB} & & & & & & & \\\\")

    # --- 2. PMF-IB ---
    for beta in IB_BETAS:
        path = f"csv/IB_Beta_{beta}/cyc_cons_IB_{beta}_aggregate.csv"
        line = extract_data_and_format(path, f"$\\beta$ = {beta}", has_lambda=True)
        latex_lines.append(line)

    latex_lines.append("        \\midrule")
    # Aggiunti gli & mancanti per allineare correttamente l'intestazione HCE + ResNet
    latex_lines.append("        \\textbf{HCE + ResNet} & & & & \\\\")

    # # --- 3. Gaussian (HCE + ResNet) ---
    # for label, folder_type in GAUSSIAN_TYPES:
    #     path = f"csv/Gaussian/{folder_type}/cyc_cons_Gaussian_{folder_type}_aggregate.csv"
    #     line = extract_data_and_format(path, label, has_lambda=False)
    #     latex_lines.append(line)

    # latex_lines.append("        \\midrule")

    # # --- 4. Conditional (CNF + ResNet) ---
    # path_cond = "csv/Conditional/cyc_cons_Conditional_aggregate.csv"
    # line_cond = extract_data_and_format(path_cond, "\\textbf{CNF + ResNet}", has_lambda=False)
    # latex_lines.append(line_cond)

    # latex_lines.append("        \\bottomrule")
    # latex_lines.append("        \\end{tabular}%")

    return "\n".join(latex_lines)

if __name__ == "__main__":
    final_latex = generate_latex_table()
    print(final_latex)

\begin{tabular}{l c c c c c c | c}
        \toprule
        \textbf{Model} & $\lambda=0.0$ & $\lambda=0.2$ & $\lambda=0.4$ & $\lambda=0.6$ & $\lambda=0.8$ & $\lambda=1.0$ & $\lambda=\tilde{\lambda}$\\
        \midrule
        \textbf{PMF-GMM} & & & & & & & \\
        Scale = 0.1 & 93.93\% $\pm$ 0.41\% & 94.10\% $\pm$ 0.33\% & 93.28\% $\pm$ 0.30\% & 90.52\% $\pm$ 0.29\% & 83.72\% $\pm$ 0.66\% & 72.69\% $\pm$ 1.07\% & 72.71\% $\pm$ 1.07\% \\
        Scale = 1.0 & 96.56\% $\pm$ 1.53\% & 96.59\% $\pm$ 1.58\% & 96.03\% $\pm$ 1.80\% & 94.68\% $\pm$ 2.10\% & 92.10\% $\pm$ 2.45\% & 87.72\% $\pm$ 2.87\% & 87.72\% $\pm$ 2.87\% \\
        Scale = 2.0 & 95.76\% $\pm$ 2.53\% & 95.83\% $\pm$ 2.32\% & 95.40\% $\pm$ 2.23\% & 94.09\% $\pm$ 2.34\% & 91.35\% $\pm$ 2.61\% & 86.82\% $\pm$ 2.79\% & 86.82\% $\pm$ 2.79\% \\
        Scale = 3.0 & 97.79\% $\pm$ 0.54\% & 97.76\% $\pm$ 0.44\% & 97.40\% $\pm$ 0.40\% & 96.39\% $\pm$ 0.46\% & 94.06\% $\pm$ 0.64\% & 90.03\% $\pm$ 0.94\% & 90.03\% $\pm$ 0.94\% \\
    